# SQL queries for filtering Table

- In the previous exercise, you have run a simple SQL query on a DataFrame. There are more sophisticated queries you can construct to obtain the result that you want and use it for downstream analysis such as data visualization and Machine Learning. In this exercise, we will use the temporary table `people` that you created previously and filter out the rows where the `"sex"` is male and female and create two DataFrames.

- Remember, you already have `SparkSession` `spark` and `people` temporary table available in your workspace.

## Instructions

- Filter the `people` table to select all rows where `sex` is `female` into `people_female_df` DataFrame.
- Filter the `people` table to select all rows where `sex` is `male` into `people_male_df` DataFrame.
- Count the number of rows in both `people_female` and `people_male` DataFrames.

In [ ]:
# Explicitly Point Spark to the Correct Python Executable
# If you already have numpy installed in a specific environment (like conda or a virtualenv), but PySpark is defaulting to /usr/bin/python3 or system Python, 
# explicitly set the Python environment variables before initializing SparkContext:

import os
import sys

# Set these before initializing SparkContext (sc)
os.environ['PYSPARK_PYTHON'] = sys.executable
os.environ['PYSPARK_DRIVER_PYTHON'] = sys.executable

In [ ]:
from pyspark.sql import SparkSession

In [ ]:
# If you are developing an application you should use local mode. 
# Running in yarn mode is bit time consuming than running in local mode
# If you still insist to run in yarn mode then make sure that
# 1. You assigned more than one processor in the VM settings
# 2. Run only one kernel at a time

In [ ]:
# spark = SparkSession. \
#     builder. \
#     enableHiveSupport(). \
#     appName('Demo'). \
#     master('yarn'). \
#     getOrCreate()

In [ ]:
spark = SparkSession. \
    builder. \
    enableHiveSupport(). \
    appName('Demo'). \
    master('local[*]'). \
    getOrCreate()

In [ ]:
sc = spark.sparkContext

In [9]:
file_path = "file:///home/talentum/UBUNTU_HOME/test-jupyter/P2/M3/SM3/3_InteractingwithDataFramesusingpysparksql/Dataset/people.csv"

# Create an DataFrame from file_path
people_df = spark.read.csv(file_path, header=True, inferSchema=True)

# Create a temporary table "people"
people_df.createOrReplaceTempView("people")

# Filter the people table to select female sex 
people_female_df = spark.sql('SELECT * FROM people WHERE sex=="female"')

# Filter the people table DataFrame to select male sex
people_male_df = spark.sql('SELECT * FROM people WHERE sex=="male"')

# Count the number of rows in both DataFrames
print("There are {} rows in the people_female_df and {} rows in the people_male_df DataFrames".format(people_female_df.count(), people_male_df.count()))

There are 49014 rows in the people_female_df and 49066 rows in the people_male_df DataFrames


In [43]:
data = [("Engineering","Alice",75000,5000),
        ("Engineering","Alice",85000,7000),
        ("Sales","Charlie",60000,3000),
        ("Sales","David",55000,4000)]

columns = ["department","name","salary","bonus"]

data_df = spark.createDataFrame(data,schema=columns)


data_df.createOrReplaceTempView("employee")

count_name_df = spark.sql('SELECT  department,count(name) as name_count from employee group by department')
count_name_df.show()

avg_sal_df = spark.sql('select department,avg(salary) as avg_sal from employee group by department')
avg_sal_df.show()

sum_bonus_df = spark.sql('select department,sum(bonus) as sum_bonus from employee group by department' )
sum_bonus_df.show()

+-----------+----------+
| department|name_count|
+-----------+----------+
|      Sales|         2|
|Engineering|         2|
+-----------+----------+

+-----------+-------+
| department|avg_sal|
+-----------+-------+
|      Sales|57500.0|
|Engineering|80000.0|
+-----------+-------+

+-----------+---------+
| department|sum_bonus|
+-----------+---------+
|      Sales|     7000|
|Engineering|    12000|
+-----------+---------+



In [47]:
data = [("Engineering","Alice",75000,5000),
        ("Engineering","Bob",85000,7000),
        ("Engineering","Alice",85000,7000),
        ("Sales","Charlie",60000,3000),
        ("Sales","David",55000,4000)]

columns = ["department","name","salary","bonus"]

data_df = spark.createDataFrame(data,schema=columns)

data_df.groupBy("department", 'name').count().show()


data_df.groupBy("department").avg("salary").show()

data_df.groupBy("department").sum("bonus").show()



result_df = data_df.groupBy("department").agg({
    "name":"count",
    "salary":"avg",
    "bonus":"sum"
})

result_df.show()

+-----------+-------+-----+
| department|   name|count|
+-----------+-------+-----+
|Engineering|  Alice|    2|
|      Sales|Charlie|    1|
|      Sales|  David|    1|
|Engineering|    Bob|    1|
+-----------+-------+-----+

+-----------+-----------------+
| department|      avg(salary)|
+-----------+-----------------+
|      Sales|          57500.0|
|Engineering|81666.66666666667|
+-----------+-----------------+

+-----------+----------+
| department|sum(bonus)|
+-----------+----------+
|      Sales|      7000|
|Engineering|     19000|
+-----------+----------+

+-----------+----------+-----------+-----------------+
| department|sum(bonus)|count(name)|      avg(salary)|
+-----------+----------+-----------+-----------------+
|      Sales|      7000|          2|          57500.0|
|Engineering|     19000|          3|81666.66666666667|
+-----------+----------+-----------+-----------------+

